---

# Sesión 07: Aprendizaje No Supervisado
## Segmentación de Clientes con K-Means

**Objetivo**: Descubrir segmentos naturales de clientes usando clustering K-Means.

---





---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de capacitar, actualizar y fortalecer las competencias en el manejo de datos con Python.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---


In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualizaciones
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Librerías cargadas correctamente")

## Paso 1: Crear Dataset de Clientes

In [ ]:
# Crear dataset simulado de clientes
np.random.seed(42)
n_customers = 500

# Generar 3 segmentos naturales con características distintas
# Segmento 1: VIP (altos ingresos, alto gasto, alta frecuencia)
n_vip = 100
vip = pd.DataFrame({
    'annual_income': np.random.normal(120000, 20000, n_vip),
    'annual_spending': np.random.normal(8000, 1500, n_vip),
    'purchase_frequency': np.random.normal(25, 5, n_vip),
    'customer_tenure_months': np.random.normal(48, 12, n_vip),
    'age': np.random.normal(45, 8, n_vip)
})

# Segmento 2: Regulares (ingresos medios, gasto medio)
n_regular = 250
regular = pd.DataFrame({
    'annual_income': np.random.normal(60000, 15000, n_regular),
    'annual_spending': np.random.normal(3000, 800, n_regular),
    'purchase_frequency': np.random.normal(12, 4, n_regular),
    'customer_tenure_months': np.random.normal(24, 10, n_regular),
    'age': np.random.normal(35, 10, n_regular)
})

# Segmento 3: Ocasionales (bajos ingresos, bajo gasto)
n_ocasional = 150
ocasional = pd.DataFrame({
    'annual_income': np.random.normal(35000, 10000, n_ocasional),
    'annual_spending': np.random.normal(1000, 400, n_ocasional),
    'purchase_frequency': np.random.normal(5, 2, n_ocasional),
    'customer_tenure_months': np.random.normal(12, 6, n_ocasional),
    'age': np.random.normal(28, 8, n_ocasional)
})

# Combinar todos los segmentos
df_customers = pd.concat([vip, regular, ocasional], ignore_index=True)
df_customers['customer_id'] = [f'CUST{i:04d}' for i in range(1, len(df_customers) + 1)]

# Asegurar valores positivos y reordenar columnas
for col in df_customers.columns:
    if col != 'customer_id':
        df_customers[col] = df_customers[col].clip(lower=0)

df_customers = df_customers[['customer_id', 'annual_income', 'annual_spending', 
                              'purchase_frequency', 'customer_tenure_months', 'age']]

print("Dataset de Clientes creado:")
print(f"Dimensiones: {df_customers.shape}")
print("\nPrimeras filas:")
print(df_customers.head())
print("\nEstadísticas descriptivas:")
print(df_customers.describe())

## Paso 2: Análisis Exploratorio de Datos (EDA)

In [ ]:
# TODO: Explora las distribuciones de las variables
# Crea histogramas para cada variable numérica

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

features = ['annual_income', 'annual_spending', 'purchase_frequency', 
            'customer_tenure_months', 'age']

for idx, col in enumerate(features):
    axes[idx].hist(df_customers[col], bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribución de {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frecuencia')
    axes[idx].grid(True, alpha=0.3)

axes[-1].axis('off')  # Ocultar el último subplot vacío
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Matriz de correlación
plt.figure(figsize=(10, 8))
correlation_matrix = df_customers[features].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1)
plt.title('Matriz de Correlación', fontsize=16, fontweight='bold')
plt.show()

print("\n💡 Observaciones:")
print("- ¿Qué variables están más correlacionadas?")
print("- ¿Tiene sentido de negocio?")

In [ ]:
# TODO: Scatter plots de pares de variables clave
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Annual Income vs Annual Spending
axes[0].scatter(df_customers['annual_income'], df_customers['annual_spending'], 
                alpha=0.5, edgecolors='k', s=50)
axes[0].set_xlabel('Ingresos Anuales ($)', fontsize=12)
axes[0].set_ylabel('Gasto Anual ($)', fontsize=12)
axes[0].set_title('Ingresos vs Gasto', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Purchase Frequency vs Annual Spending
axes[1].scatter(df_customers['purchase_frequency'], df_customers['annual_spending'], 
                alpha=0.5, edgecolors='k', s=50)
axes[1].set_xlabel('Frecuencia de Compra (visitas/mes)', fontsize=12)
axes[1].set_ylabel('Gasto Anual ($)', fontsize=12)
axes[1].set_title('Frecuencia vs Gasto', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 ¿Puedes ver agrupaciones naturales en los datos?")

## Paso 3: Preparación de Datos - Feature Scaling

In [ ]:
# TODO: Prepara los datos para K-Means
# 1. Selecciona solo las features numéricas (excluye customer_id)
# 2. Aplica StandardScaler

X = df_customers[features].values

print("Datos ANTES del scaling:")
print(f"Media: {X.mean(axis=0)}")
print(f"Desviación estándar: {X.std(axis=0)}")

# Aplicar StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nDatos DESPUÉS del scaling:")
print(f"Media: {X_scaled.mean(axis=0).round(10)}")
print(f"Desviación estándar: {X_scaled.std(axis=0)}")

print("\n✓ Feature scaling completado")

## Paso 4: Determinar K Óptimo - Elbow Method

In [ ]:
# TODO: Implementa el Elbow Method
# Prueba K de 1 a 10 y grafica la inertia

inertias = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Graficar el Elbow
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, marker='o', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (K)', fontsize=12)
plt.ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
plt.title('Elbow Method para Determinar K Óptimo', fontsize=16, fontweight='bold')
plt.xticks(K_range)
plt.grid(True, alpha=0.3)
plt.show()

print("💡 Busca el 'codo' donde la reducción de inertia disminuye significativamente")

## Paso 5: Determinar K Óptimo - Silhouette Score

In [ ]:
# TODO: Calcula Silhouette Score para K de 2 a 10
silhouette_scores = []
K_range_silhouette = range(2, 11)

for k in K_range_silhouette:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"K = {k}: Silhouette Score = {score:.4f}")

# Graficar Silhouette Scores
plt.figure(figsize=(10, 6))
plt.plot(K_range_silhouette, silhouette_scores, marker='o', linewidth=2, markersize=8, color='green')
plt.xlabel('Número de Clusters (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Score para Diferentes K', fontsize=16, fontweight='bold')
plt.xticks(K_range_silhouette)
plt.grid(True, alpha=0.3)
plt.axhline(y=0.5, color='r', linestyle='--', label='Umbral 0.5')
plt.legend()
plt.show()

best_k = K_range_silhouette[np.argmax(silhouette_scores)]
print(f"\n✨ K óptimo según Silhouette Score: {best_k}")

## Paso 6: Entrenar K-Means con K Óptimo

In [ ]:
# TODO: Entrena el modelo K-Means con el K óptimo
# Usa el K que encontraste (probablemente 3)

optimal_k = 3  # Ajusta según tu análisis

kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled)

# Agregar labels al dataset original
df_customers['cluster'] = cluster_labels

print(f"✓ Modelo K-Means entrenado con K = {optimal_k}")
print(f"\nDistribución de clusters:")
print(df_customers['cluster'].value_counts().sort_index())
print(f"\nSilhouette Score final: {silhouette_score(X_scaled, cluster_labels):.4f}")

## Paso 7: Análisis de Perfiles de Clusters

In [ ]:
# TODO: Analiza las características promedio de cada cluster
cluster_profiles = df_customers.groupby('cluster')[features].mean()

print("📊 PERFILES DE CLUSTERS")
print("=" * 80)
print(cluster_profiles.round(2))
print("\n")

# Contar miembros por cluster
cluster_counts = df_customers['cluster'].value_counts().sort_index()
print("\n📊 TAMAÑO DE CLUSTERS")
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(df_customers)) * 100
    print(f"Cluster {cluster_id}: {count} clientes ({percentage:.1f}%)")

In [ ]:
# TODO: Visualiza los perfiles con gráficos de barras
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, feature in enumerate(features):
    cluster_profiles[feature].plot(kind='bar', ax=axes[idx], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    axes[idx].set_title(f'{feature} por Cluster', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Cluster')
    axes[idx].set_ylabel('Promedio')
    axes[idx].grid(axis='y', alpha=0.3)
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=0)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

## Paso 8: Visualización 2D con PCA

In [ ]:
# TODO: Reduce dimensionalidad a 2D con PCA y visualiza clusters
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Crear DataFrame para visualización
df_pca = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'Cluster': cluster_labels
})

# Obtener centroides en espacio PCA
centroids = kmeans_final.cluster_centers_
centroids_pca = pca.transform(centroids)

# Visualizar
plt.figure(figsize=(12, 8))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for cluster_id in range(optimal_k):
    cluster_data = df_pca[df_pca['Cluster'] == cluster_id]
    plt.scatter(cluster_data['PC1'], cluster_data['PC2'], 
                c=colors[cluster_id], label=f'Cluster {cluster_id}',
                alpha=0.6, edgecolors='k', s=100)

# Plotear centroides
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], 
            c='black', marker='X', s=500, edgecolors='yellow', linewidths=2,
            label='Centroides')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)', fontsize=12)
plt.title('Visualización de Clusters en 2D (PCA)', fontsize=16, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n💡 Varianza explicada por 2 componentes: {pca.explained_variance_ratio_.sum()*100:.1f}%")

## Paso 9: Nombrar y Describir Clusters

In [ ]:
# TODO: Basándote en los perfiles, asigna nombres descriptivos a los clusters
# Analiza las características y determina qué tipo de cliente representa cada cluster

cluster_names = {
    0: "Nombre del Cluster 0",  # Tu análisis aquí
    1: "Nombre del Cluster 1",  # Tu análisis aquí
    2: "Nombre del Cluster 2"   # Tu análisis aquí
}

df_customers['cluster_name'] = df_customers['cluster'].map(cluster_names)

print("📛 NOMBRES Y DESCRIPCIONES DE CLUSTERS")
print("=" * 80)

for cluster_id in range(optimal_k):
    print(f"\nCLUSTER {cluster_id}: {cluster_names[cluster_id]}")
    print("-" * 80)
    cluster_data = df_customers[df_customers['cluster'] == cluster_id][features]
    print(cluster_data.describe().loc[['mean', 'std']].round(2))
    print(f"\nCaracterísticas distintivas:")
    # TODO: Escribe las características distintivas de cada cluster

## Paso 10: Insights y Recomendaciones de Negocio

In [ ]:
# TODO: Genera insights accionables para cada cluster

print("💼 ESTRATEGIAS DE MARKETING POR SEGMENTO")
print("=" * 80)

print("\n📊 CLUSTER 0: [Nombre]")
print("Características: [Describe características principales]")
print("Estrategia sugerida:")
print("  - Acción 1: [Tu recomendación]")
print("  - Acción 2: [Tu recomendación]")
print("  - Acción 3: [Tu recomendación]")

print("\n📊 CLUSTER 1: [Nombre]")
print("Características: [Describe características principales]")
print("Estrategia sugerida:")
print("  - Acción 1: [Tu recomendación]")
print("  - Acción 2: [Tu recomendación]")
print("  - Acción 3: [Tu recomendación]")

print("\n📊 CLUSTER 2: [Nombre]")
print("Características: [Describe características principales]")
print("Estrategia sugerida:")
print("  - Acción 1: [Tu recomendación]")
print("  - Acción 2: [Tu recomendación]")
print("  - Acción 3: [Tu recomendación]")

## 🎯 Ejercicio Bonus: Silhouette Analysis Detallado

In [ ]:
# Análisis de silhouette por muestra para visualizar calidad de clustering
from matplotlib import cm

silhouette_vals = silhouette_samples(X_scaled, cluster_labels)

fig, ax = plt.subplots(figsize=(10, 8))
y_lower = 10

for i in range(optimal_k):
    cluster_silhouette_vals = silhouette_vals[cluster_labels == i]
    cluster_silhouette_vals.sort()
    
    size_cluster_i = cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    color = cm.nipy_spectral(float(i) / optimal_k)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_vals,
                      facecolor=color, edgecolor=color, alpha=0.7)
    
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, f'Cluster {i}')
    y_lower = y_upper + 10

ax.set_title('Silhouette Plot por Cluster', fontsize=16, fontweight='bold')
ax.set_xlabel('Silhouette Coefficient', fontsize=12)
ax.set_ylabel('Cluster', fontsize=12)
ax.axvline(x=silhouette_score(X_scaled, cluster_labels), color='red', linestyle='--',
           label=f'Promedio: {silhouette_score(X_scaled, cluster_labels):.3f}')
ax.legend()
plt.show()

print("\n💡 Interpretación:")
print("- Valores > promedio: Bien asignados")
print("- Valores < promedio: Posibles asignaciones incorrectas")
print("- Grosor de cada sección: Tamaño del cluster")

## 🎓 Reflexión Final

**Responde las siguientes preguntas**:

1. **Sobre K-Means**:
   - ¿Por qué es crítico el feature scaling en K-Means?
   - ¿Qué pasaría si una feature tuviera un rango 1000x mayor que otra?

2. **Sobre la elección de K**:
   - ¿El Elbow Method y Silhouette Score sugirieron el mismo K?
   - ¿Cómo decidirías si hay conflicto entre métodos?

3. **Sobre los clusters**:
   - ¿Los clusters encontrados tienen sentido de negocio?
   - ¿Qué cluster representa mayor oportunidad de crecimiento?
   - ¿Algún cluster está en riesgo de churn?

4. **Sobre aplicaciones**:
   - ¿En qué otros problemas de negocio aplicarías clustering?
   - ¿Cuándo NO deberías usar K-Means?

---
**¡Excelente trabajo!** Has dominado K-Means y segmentación de clientes. En la Sesión 08, darás un salto gigante hacia el futuro con Transformers y LLMs. 🚀🤖